In [47]:
# Cell 1 - imports & config
import os
import uuid
import json
import base64
from pathlib import Path
import pandas as pd
import numpy as np
import csv
from datetime import datetime

# Paths
INPUT_DIR = Path(r"C:\static_inference\data")
OUTPUT_DIR = Path(r"C:\static_inference\data\output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODULE_INF_FILES = {
    "battery":  OUTPUT_DIR / "battery_inference.csv",
    "body":     OUTPUT_DIR / "body_inference.csv",
    "engine":   OUTPUT_DIR / "engine_inference.csv",
    "transmission": OUTPUT_DIR / "transmission_inference.csv",
    "tyre":     OUTPUT_DIR / "tyre_inference.csv",
}

OUT_CSV = OUTPUT_DIR / "vehicle_alerts.csv"
OUT_XLSX = OUTPUT_DIR / "vehicle_alerts.xlsx"   # optional

# thresholds (tune as you like)
WARNING_TH = 0.50   # composite >= this -> warning (severity 1)
ANOMALY_TH = 0.75   # composite >= this -> anomaly/critical (severity 2 or 3)
CRITICAL_TH = 0.90  # composite >= this -> critical (severity 3)
# you can decide to use ANOMALY_TH=0.75 as in your previous logic; critical higher if needed

# fields we expect in module inference csvs (we'll be defensive)
EXPECTED_MIN_COLS = ["row_hash","timestamp","date","source_id","composite_score","anomaly_severity","explain_top_k_json","dense_per_feature_error_json"]


In [48]:
# Cell 2 - helpers
def to_b64_json_safe(obj):
    """
    Take obj (dict/list/str) and return a base64-encoded JSON string.
    If obj already looks like a base64 string (only base64 chars) we return it unchanged.
    """
    if obj is None:
        return ""
    if isinstance(obj, str):
        s = obj.strip()
        # quick heuristic: if it's already base64 JSON (our pipeline uses base64)
        try:
            if all(c in "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/=" for c in s) and len(s) % 4 in (0,):
                # try decode to see if it's JSON
                decoded = base64.b64decode(s)
                try:
                    json.loads(decoded.decode("utf-8"))
                    return s
                except Exception:
                    pass
        except Exception:
            pass
        # not a base64 JSON -> try parse as JSON string
        try:
            parsed = json.loads(s)
            raw = json.dumps(parsed, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
            return base64.b64encode(raw).decode("ascii")
        except Exception:
            # treat as raw string: encode as JSON string then base64
            raw = json.dumps(s, ensure_ascii=False).encode("utf-8")
            return base64.b64encode(raw).decode("ascii")

    # for dict/list/number etc
    try:
        raw = json.dumps(obj, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
        return base64.b64encode(raw).decode("ascii")
    except Exception:
        # fallback: convert to str then base64
        raw = str(obj).encode("utf-8")
        return base64.b64encode(raw).decode("ascii")

def short_alert_id():
    # short but reasonably unique id; 16 hex chars
    return uuid.uuid4().hex[:16]

def safe_dt(x):
    # Accept strings or pandas timestamps. Return ISO 8601 str or empty.
    if pd.isna(x):
        return ""
    if isinstance(x, (pd.Timestamp, datetime)):
        # maintain tz-aware if present; convert to ISO
        try:
            return x.isoformat()
        except Exception:
            return str(x)
    # try parseable string
    try:
        return pd.to_datetime(x).isoformat()
    except Exception:
        return str(x)


In [49]:
# Cell 3 - read module CSVs defensively and stack
dfs = []
missing = []
for module, p in MODULE_INF_FILES.items():
    if not p.exists():
        missing.append(module)
        continue
    # read with engine='python' for robustness
    df = pd.read_csv(p, dtype=str, keep_default_na=False, na_values=[""], engine="python")
    df.columns = [c.strip() for c in df.columns.tolist()]  # normalize
    df["__module"] = module
    # ensure core columns exist
    for c in EXPECTED_MIN_COLS:
        if c not in df.columns:
            df[c] = ""
    dfs.append(df)

if missing:
    raise FileNotFoundError(f"Missing module inference files for: {missing}")

all_df = pd.concat(dfs, ignore_index=True, sort=False)
print("Read rows:", {m: len(df) for m, df in zip(MODULE_INF_FILES.keys(), dfs)})
print("Stacked rows:", len(all_df))
# quick dtype fixes for composite_score
all_df["composite_score"] = pd.to_numeric(all_df["composite_score"], errors="coerce")
all_df["anomaly_severity"] = pd.to_numeric(all_df["anomaly_severity"], errors="coerce").astype("Int64")


Read rows: {'battery': 31799, 'body': 31799, 'engine': 31799, 'transmission': 31799, 'tyre': 31799}
Stacked rows: 158995


In [50]:
# Cell 4 - candidate alerts (one row per inference row that crosses thresholds)
cand_rows = []

for idx, r in all_df.iterrows():
    comp = r.get("composite_score")
    # skip missing numeric composite
    if pd.isna(comp):
        continue
    # determine severity according to thresholds (map to your label mapping)
    if comp >= CRITICAL_TH:
        sev = 3
        sev_lbl = "critical"
    elif comp >= ANOMALY_TH:
        sev = 2
        sev_lbl = "anomaly"
    elif comp >= WARNING_TH:
        sev = 1
        sev_lbl = "warning"
    else:
        # do not create alert for normal rows
        continue

    alert_id = short_alert_id()
    vehicle_id = r.get("source_id") or r.get("kafka_key") or r.get("vehicle_id") or ""
    ts = safe_dt(r.get("timestamp") or r.get("inference_timestamp") or r.get("inference_ts"))
    date = r.get("date") or (pd.to_datetime(ts).date().isoformat() if ts else "")
    top_features_b64 = to_b64_json_safe(r.get("explain_top_k_json") or r.get("explain_top_k") or "")
    dense_errors_b64 = to_b64_json_safe(r.get("dense_per_feature_error_json") or "")
    linked_row_hashes_b64 = to_b64_json_safe([r.get("row_hash")] if r.get("row_hash") else [])

    cand_rows.append({
        "alert_id": alert_id,
        "vehicle_id": vehicle_id,
        "module": r["__module"],
        "start_ts": ts,
        "end_ts": ts,
        "duration_s": 0,
        "n_points": 1,
        "peak_composite": float(comp),
        "mean_composite": float(comp),
        "severity": int(sev),
        "top_features_b64": top_features_b64,
        "dense_errors_b64": dense_errors_b64,
        "linked_row_hashes_b64": linked_row_hashes_b64,
        "date": date,
        "severity_label": sev_lbl,
        # preserve raw row_hash for debugging (not complex)
        "row_hash": r.get("row_hash") or ""
    })

cand_df = pd.DataFrame(cand_rows)
print("Candidate alerts:", len(cand_df))


Candidate alerts: 71499


In [52]:
# Cell 5 - simple grouping: merge adjacent per-module alerts if contiguous within GAP seconds
# Parameters
GAP_S = 300   # if two alert rows for same vehicle+module within 300s, merge into one alert window
cand_df = cand_df.sort_values(["vehicle_id","module","start_ts"]).reset_index(drop=True)

# helper to to datetime
def to_ts(x):
    try:
        return pd.to_datetime(x)
    except Exception:
        return pd.NaT

cand_df["_ts"] = cand_df["start_ts"].apply(to_ts)

merged = []
for (veh, mod), g in cand_df.groupby(["vehicle_id","module"], sort=False):
    g = g.sort_values("_ts").reset_index(drop=True)
    if g.empty:
        continue
    cur = g.loc[0].to_dict()
    cur["_start"] = cur["_ts"]
    cur["_end"] = cur["_ts"]
    cur["n_points"] = int(cur["n_points"])
    for i in range(1, len(g)):
        row = g.loc[i].to_dict()
        if pd.isna(row["_ts"]) or pd.isna(cur["_end"]):
            # can't compare, flush previous and start new
            merged.append(cur)
            cur = row
            cur["_start"] = cur["_ts"]; cur["_end"] = cur["_ts"]
            continue
        gap = (row["_ts"] - cur["_end"]).total_seconds()
        if gap <= GAP_S:
            # merge: extend end, update peak/mean/n_points, combine linked rows and top features
            cur["_end"] = row["_ts"]
            cur["n_points"] = int(cur.get("n_points",1)) + int(row.get("n_points",1))
            cur["peak_composite"] = max(cur.get("peak_composite",0.0), row.get("peak_composite",0.0))
            cur["mean_composite"] = (cur.get("mean_composite",0.0)* (cur["n_points"]-int(row.get("n_points",1))) + row.get("mean_composite",0.0)*int(row.get("n_points",1))) / max(1, cur["n_points"])
            # combine linked hashes (maintain base64 list)
            lhs = []
            try:
                lhs = json.loads(base64.b64decode(cur["linked_row_hashes_b64"]).decode("utf-8"))
            except Exception:
                lhs = []
            rhs = []
            try:
                rhs = json.loads(base64.b64decode(row["linked_row_hashes_b64"]).decode("utf-8"))
            except Exception:
                rhs = []
            uni = list(dict.fromkeys(lhs + rhs))
            cur["linked_row_hashes_b64"] = base64.b64encode(json.dumps(uni, separators=(",",":")).encode("utf-8")).decode("ascii")
            # combine top_features: we don't decode semantics here; just keep concatenation of base64 blobs
            cur["top_features_b64"] = cur.get("top_features_b64","") + "|" + row.get("top_features_b64","")
            # keep severity = max
            cur["severity"] = max(int(cur.get("severity",0)), int(row.get("severity",0)))
            cur["severity_label"] = "critical" if cur["severity"]>=3 else ("anomaly" if cur["severity"]==2 else "warning")
        else:
            # flush current
            merged.append(cur)
            cur = row
            cur["_start"] = cur["_ts"]; cur["_end"] = cur["_ts"]
    merged.append(cur)

merged_df = pd.DataFrame(merged).reset_index(drop=True)
# finalize fields
merged_df["start_ts"] = merged_df["_start"].apply(lambda x: x.isoformat() if not pd.isna(x) else "")
merged_df["end_ts"]   = merged_df["_end"].apply(lambda x: x.isoformat() if not pd.isna(x) else "")
merged_df["duration_s"] = (merged_df["_end"] - merged_df["_start"]).dt.total_seconds().fillna(0).astype(int)
# ensure types and columns order
final_cols = ["alert_id","vehicle_id","module","start_ts","end_ts","duration_s","n_points",
              "peak_composite","mean_composite","severity","top_features_b64","dense_errors_b64","linked_row_hashes_b64","date","severity_label","row_hash"]
out_df = merged_df[final_cols].copy()
print("Merged alerts:", len(out_df))


Merged alerts: 25
